# Fault-tolerant multi-zone irrigation: closed-loop fault-injection simulation (v9, notebook)

Single zone, three redundant soil-moisture probes, FAO-56 water balance (tomato parameters), ET0 from Hargreaves-Samani.
Compares: `oracle` (true moisture), `single` (one probe), `median`, `vote` (strict majority inside the zone) `reliab_v7` (reliability-weighted fusion + exclusion + model-based safe mode with innovation gate + a drift indicator based on each probe's residual against the other probes, tracked against its own commissioning baseline) and `reliab` (v8: the same plus commissioning self-calibration of the probe offsets). `*_adapt` variants use FAO-56 adaptive thresholds; `s1_timer` and `s3_et0` use no sensors.

**How to run:** Run All. Only `numpy` is required; `scipy`, `pandas`, `matplotlib` are used if present.
**Weather:** put a NASA POWER daily CSV (`T2M_MAX`, `T2M_MIN`) next to the notebook as `weather_hillah.csv`, or leave `TRY_DOWNLOAD = True` to download it when the machine has internet. With `REQUIRE_REAL_WEATHER = True` (default) the notebook stops with diagnostics if real weather cannot be parsed, instead of silently using synthetic data. Run the notebook as provided: do not paste other weather cells above it.
**v9 fix:** in v8 a probe with a large offset was excluded during the commissioning cycles (before its offset was learned), the two commissioning counters then went out of step and the drift baseline was contaminated, so a healthy probe stayed flagged (false-alarm rate 3.3% instead of about 0 in two seeds of 30). In v9 no probe is excluded while the offsets are learned and the drift statistic restarts on offset-corrected readings.
**v8 addition:** commissioning self-calibration. Each probe's offset relative to the group mean is learned in the first N0 cycles after installation and removed from its readings, so that calibration offsets between healthy probes are not mistaken for faults and cannot bias the fused estimate near field capacity (found in v7 development: a healthy probe with a positive offset was excluded and the remaining probes could not reach the stop threshold). It assumes the probes are healthy during the first N0 cycles. `reliab_v7` and `reliab_adapt_v7` keep the v7 algorithm for comparison.
**v7 additions:** sensor-free baselines `s1_timer` (fixed schedule) and `s3_et0` (ET0 water-balance scheduling), FAO-56 adaptive thresholds (`*_adapt` controllers, 2x2 ablation: fusion x thresholds), a systematic evapotranspiration error between the controller's estimate and the true demand (`ET_ERR_SD`), the metric `below_RAW_h` (hours below the FAO-56 adjusted threshold), tail metrics (P95 drainage, share of runs with drainage > 5 mm) and per-seed export. Setting `ET_ERR_SD = 0` and ignoring the new controllers reproduces v6.
**Protocol:** `RUN_MODE = "dev"` uses development years 2020-2023 and seeds 0-29. `RUN_MODE = "test"` uses the held-out years 2024-2025 and seeds 1000-1099 and is locked until `FINAL_TEST_CONFIRMED = True` is set by hand; run it once, after the algorithm and its parameters are frozen. The settings of each run are saved to `run_config_<mode>.json`.
**Status:** exploratory. Sensor noise, faults and (without a CSV) weather are synthetic; results are not experimental evidence.

In [ ]:
import os, csv, math, datetime as dt
import numpy as np

try:
    from scipy.stats import wilcoxon
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False
try:
    import pandas as pd
except Exception:
    pd = None
try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

# ---------------- settings ----------------
RUN_MODE = "dev"                # "dev": development years/seeds | "test": held-out years, run ONCE after freezing the algorithm
FINAL_TEST_CONFIRMED = False    # set True by hand only when the algorithm and parameters are frozen
DEV_YEARS, TEST_YEARS = [2020, 2021, 2022, 2023], [2024, 2025]
if RUN_MODE == "dev":
    SEED0, N_SEEDS, YEARS = 0, 30, DEV_YEARS
elif RUN_MODE == "test":
    if not FINAL_TEST_CONFIRMED:
        raise RuntimeError("Final-test mode is locked: freeze the algorithm first, then set FINAL_TEST_CONFIRMED = True.")
    SEED0, N_SEEDS, YEARS = 1000, 100, TEST_YEARS
else:
    raise ValueError("RUN_MODE must be 'dev' or 'test'")
DAYS = 20                 # simulated days per run
ET_ERR_SD = 0.10          # sd of the systematic error between the controller's ET estimate and true demand (0 = no error)
PERIOD_D = 3              # S1 fixed schedule: irrigate every PERIOD_D days at 06:00
SPD = 96                  # 15-minute steps per day
ONSET = 5 * SPD           # fault starts on day 5
LAT, LON = 32.48, 44.42   # Hillah
WEATHER_CSV = "weather_hillah.csv"
TRY_DOWNLOAD = True
REQUIRE_REAL_WEATHER = True   # stop instead of silently falling back to synthetic weather
MONTHS = [6, 7, 8, 9]
OUT_DIR = "sim_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
print("scipy:", HAVE_SCIPY, "| pandas:", pd is not None, "| matplotlib:", plt is not None)

In [ ]:
# ---------------- agronomic parameters (assumed; verify against FAO-56 tables) ----------------
FC, PWP, ZR = 0.30, 0.15, 0.5        # field capacity, wilting point, root depth (m)
P = 0.40                              # allowable depletion, tomato
TL = FC - P * (FC - PWP)              # start-irrigation threshold (fixed here, to isolate the fusion rule)
DTH = 0.04                            # hysteresis band
FC_MARGIN = 0.015                     # adaptive stop threshold <= FC - FC_MARGIN (must exceed the probe error)
TH = TL + DTH                         # stop threshold
DT_H, RATE, KC = 0.25, 4.0, 1.15      # step (h), irrigation rate (mm/h), tomato Kc mid-season
N = 3                                 # probes per zone
W, LAM, GAM, RMIN, GATE = 8, 0.9, 3.0, 0.5, 0.05   # reliability parameters (to be tuned on development data only)
ALPHA_F, ALPHA_S, N0, H_D, RATIO, W4 = 1/32, 1/1920, 96, 0.006, 1.5, 0.4   # v6 drift indicator: fast/slow EWMA, commissioning cycles, threshold, uniqueness ratio, weight
print(f"TL={TL:.3f}  TH={TH:.3f}")

In [ ]:
# ---------------- weather and ET0 ----------------
def ra_mm(doy, lat=LAT):
    """FAO-56 eq. 21 extraterrestrial radiation in mm/day equivalent evaporation (MJ x 0.408)."""
    doy = np.asarray(doy, dtype=float)
    phi = np.radians(lat)
    dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    dec = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(dec), -1, 1))
    ra = (24 * 60 / np.pi) * 0.0820 * dr * (ws * np.sin(phi) * np.sin(dec) + np.cos(phi) * np.cos(dec) * np.sin(ws))
    return 0.408 * ra

def hargreaves(tmax, tmin, doy):
    tmax, tmin = np.asarray(tmax, dtype=float), np.asarray(tmin, dtype=float)
    tmean = (tmax + tmin) / 2
    return np.maximum(0.0023 * (tmean + 17.8) * np.sqrt(np.maximum(tmax - tmin, 0)) * ra_mm(doy), 0)

def download_nasa_power(path, start="20200101", end="20251231"):
    import urllib.request
    url = ("https://power.larc.nasa.gov/api/temporal/daily/point?parameters=T2M_MAX,T2M_MIN&community=AG"
           f"&longitude={LON}&latitude={LAT}&start={start}&end={end}&format=CSV")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "SmartIrrigationResearch/1.0"})
        with urllib.request.urlopen(req, timeout=90) as r:
            data = r.read().decode("utf-8")
        if "T2M_MAX" not in data:
            print("Unexpected NASA POWER response:", data[:300]); return False
        with open(path, "w", encoding="utf-8") as f:
            f.write(data)
        print("NASA POWER download successful ->", path)
        return True
    except Exception as e:
        print("NASA POWER download failed:", e)
        return False

def load_nasa_csv(path):
    """Robust NASA POWER daily CSV reader: finds the header row itself (blank lines or extra lines after
    -END HEADER- do not matter), reads YEAR + DOY or MO/DY, rejects fill values, and reports why rows were dropped."""
    lines = open(path, encoding="utf-8-sig").read().splitlines()
    h = next((i for i, l in enumerate(lines) if "YEAR" in l.upper() and "T2M_MAX" in l.upper() and "T2M_MIN" in l.upper()), -1)
    if h < 0:
        raise ValueError("Header row with YEAR, T2M_MAX, T2M_MIN not found. First lines of the file:\n" + "\n".join(lines[:25]))
    cols = [c.strip().upper() for c in lines[h].split(",")]
    ix = {c: k for k, c in enumerate(cols)}
    rej = {"short_row": 0, "unparsable": 0, "fill_or_range": 0, "other_month": 0}
    rec = []
    for l in lines[h + 1:]:
        if not l.strip():
            continue
        f = [x.strip() for x in l.split(",")]
        if len(f) < len(cols):
            rej["short_row"] += 1; continue
        try:
            y = int(f[ix["YEAR"]])
            if "DOY" in ix:
                d = dt.date(y, 1, 1) + dt.timedelta(days=int(float(f[ix["DOY"]])) - 1)
            else:
                d = dt.date(y, int(f[ix["MO"]]), int(f[ix["DY"]]))
            a, b = float(f[ix["T2M_MAX"]]), float(f[ix["T2M_MIN"]])
        except Exception:
            rej["unparsable"] += 1; continue
        if not (a > -90 and b > -90 and a >= b and -60 < a < 65 and -60 < b < 55):
            rej["fill_or_range"] += 1; continue
        if d.month not in MONTHS:
            rej["other_month"] += 1; continue
        rec.append((d, a, b, d.timetuple().tm_yday))
    diag = {"columns": cols, "kept": len(rec), "rejected": rej}
    if not rec:
        raise ValueError("No valid observations after parsing. Diagnostics: " + str(diag) + "\nFirst lines:\n" + "\n".join(lines[:25]))
    rec.sort(key=lambda r: r[0])
    return rec, diag

def synthetic_weather():
    r = np.random.default_rng(12345)
    i = np.arange(122); doy = 152 + i
    tmax = 40 + 4 * np.sin(np.pi * i / 121) + r.normal(0, 1.5, 122)
    tmin = tmax - (15 + r.normal(0, 1.5, 122))
    dates = [dt.date(2021, 6, 1) + dt.timedelta(days=int(k)) for k in i]
    return hargreaves(tmax, tmin, doy), dates

def load_weather():
    if TRY_DOWNLOAD and not os.path.exists(WEATHER_CSV):
        download_nasa_power(WEATHER_CSV)
    if os.path.exists(WEATHER_CSV):
        try:
            rec, diag = load_nasa_csv(WEATHER_CSV)
            dates = [r[0] for r in rec]
            et0 = hargreaves([r[1] for r in rec], [r[2] for r in rec], [r[3] for r in rec])
            return et0, dates, f"REAL (NASA POWER) {WEATHER_CSV}: {len(rec)} days, {dates[0]} to {dates[-1]}", diag
        except Exception as e:
            print("Weather file problem:", e)
    if REQUIRE_REAL_WEATHER:
        raise RuntimeError("Real weather could not be loaded and REQUIRE_REAL_WEATHER=True. "
                           "Fix the file (see messages above; delete a bad weather_hillah.csv to re-download) "
                           "or set REQUIRE_REAL_WEATHER=False to run on clearly labelled SYNTHETIC weather.")
    et0, dates = synthetic_weather()
    return et0, dates, "SYNTHETIC Hillah-like fallback (NOT real weather)", {}

ET0_SERIES, DATES, WEATHER_SRC, WEATHER_DIAG = load_weather()

# windows must be runs of consecutive calendar days (months are filtered, so seasons are separated by gaps)
VALID_STARTS = [s for s in range(len(ET0_SERIES) - DAYS + 1)
                if (DATES[s + DAYS - 1] - DATES[s]).days == DAYS - 1
                and DATES[s].year in YEARS and DATES[s + DAYS - 1].year in YEARS]
if not VALID_STARTS:
    raise RuntimeError(f"No window of {DAYS} consecutive days in years {YEARS} and months {MONTHS}; check the weather file years, MONTHS and DAYS.")

def weather_window(seed):
    rw = np.random.default_rng(seed + 10_000)
    st = VALID_STARTS[int(rw.integers(0, len(VALID_STARTS)))]
    return ET0_SERIES[st:st + DAYS]

print("Weather:", WEATHER_SRC)
print("Run mode:", RUN_MODE, "| years:", YEARS)
print("Parse diagnostics:", WEATHER_DIAG)
print("Mean ET0 = %.2f mm/day (min %.2f, max %.2f) | window = %d days | valid windows = %d" %
      (ET0_SERIES.mean(), ET0_SERIES.min(), ET0_SERIES.max(), DAYS, len(VALID_STARTS)))

In [ ]:
# ---------------- controllers ----------------
CUR = {"TL": TL, "TH": TH}   # thresholds in force (fixed, or FAO-56 adaptive when the controller is a *_adapt variant)

def adaptive_thresholds(etc_day):
    p_adj = float(np.clip(P + 0.04 * (5 - etc_day), 0.1, 0.8))
    tl = FC - p_adj * (FC - PWP)
    # the stop threshold must stay below field capacity by a margin larger than the probe error; otherwise a probe with a
    # negative offset never reads 'wet enough' and the valve stays open (found in v7 development)
    return tl, max(min(FC - FC_MARGIN, tl + DTH), tl + 0.01)

def hyst(v, x):
    return 1 if x <= CUR["TL"] else (0 if x >= CUR["TH"] else v)

class Oracle:
    def __init__(s): s.v = 0
    def decide(s, r, th): s.v = hyst(s.v, th)
    def update(s, irr, et): pass

class Single:
    def __init__(s): s.v = 0
    def decide(s, r, th):
        if not np.isnan(r[0]): s.v = hyst(s.v, r[0])
    def update(s, irr, et): pass

class Median:
    def __init__(s): s.v = 0
    def decide(s, r, th):
        x = r[~np.isnan(r)]
        if len(x): s.v = hyst(s.v, np.median(x))
    def update(s, irr, et): pass

class Vote:   # strict majority of valid readings inside the zone
    def __init__(s): s.v = 0
    def decide(s, r, th):
        x = r[~np.isnan(r)]; k = len(x)
        if k == 0: return
        if (x <= CUR["TL"]).sum() > k / 2: s.v = 1
        elif (x >= CUR["TH"]).sum() > k / 2: s.v = 0
    def update(s, irr, et): pass

class Rel5:    # v5: reliability-weighted fusion + exclusion + model-based degraded mode with innovation gate
    def __init__(s):
        s.R = np.ones(N); s.buf = np.full((W, N), np.nan); s.prev = np.full(N, np.nan)
        s.v = 0; s.tm = 0.28; s.flags = np.zeros(N, bool); s.degraded = False; s.dis = 0
    def decide(s, r, th):
        s.buf = np.roll(s.buf, -1, axis=0); s.buf[-1] = r
        valid = ~np.isnan(r); e = np.ones(N)
        if valid.sum() >= 2:
            M = np.median(r[valid]); mad = np.median(np.abs(r[valid] - M))
            D = np.abs(r - M) / (1.4826 * mad + 0.01)
        else:
            D = np.zeros(N)
        for i in range(N):
            if not valid[i]: continue
            full = not np.isnan(s.buf[:, i]).any()
            S = 1.0 if (full and np.ptp(s.buf[:, i]) == 0) else 0.0
            T = 1.0 if (not np.isnan(s.prev[i]) and abs(r[i] - s.prev[i]) > 0.03) else 0.0
            e[i] = 0.5 * min(D[i] / 3, 1) + 0.3 * S + 0.2 * T
        s.R = LAM * s.R + (1 - LAM) * np.exp(-GAM * e)
        s.prev = r.copy()
        ok = valid & (s.R >= RMIN); s.flags = ~ok
        if ok.sum() >= 2:
            th_hat = float((s.R[ok] * r[ok]).sum() / s.R[ok].sum())
            if abs(th_hat - s.tm) < GATE or s.dis >= 48:
                s.tm = th_hat; s.dis = 0; s.degraded = False
            else:
                s.dis += 1; s.degraded = True
        else:
            s.degraded = True
        s.v = hyst(s.v, s.tm)
    def update(s, irr, et):
        s.tm = min(FC, s.tm + (irr - 1.1 * et) / (1000 * ZR))

class Rel7(Rel5):   # v6/v7: v5 + drift indicator
    # Each probe's residual is taken against the median of the OTHER probes, smoothed by a fast EWMA, and compared with
    # its own baseline learned during the first N0 cycles after commissioning. The statistic is only updated while all
    # probes are valid and trusted, so the reference set never changes and a constant offset cannot look like drift.
    def __init__(s):
        super().__init__()
        s.n = 0; s.zf = np.zeros(N); s.base = np.zeros(N); s.acc = np.zeros(N); s.dr = np.zeros(N)
    def decide(s, r, th):
        valid = ~np.isnan(r); trusted = valid & (s.R >= RMIN)
        if valid.all() and trusted.all():
            res = np.array([r[i] - np.median([r[k] for k in range(N) if k != i]) for i in range(N)])
            s.n += 1
            if s.n <= N0:
                s.acc += res; s.zf = s.acc / s.n
                if s.n == N0: s.base = s.acc / N0
            else:
                s.zf = (1 - ALPHA_F) * s.zf + ALPHA_F * res
                s.base = (1 - ALPHA_S) * s.base + ALPHA_S * s.zf
        st = s.zf - s.base if s.n >= N0 else np.zeros(N)
        a = np.abs(st); s.dr = np.zeros(N)
        for i in range(N):
            others = max([a[k] for k in range(N) if k != i], default=0.0)
            if a[i] > H_D and a[i] >= RATIO * others: s.dr[i] = 1.0
        s.buf = np.roll(s.buf, -1, axis=0); s.buf[-1] = r
        e = np.ones(N)
        if valid.sum() >= 2:
            M = np.median(r[valid]); mad = np.median(np.abs(r[valid] - M)); D = np.abs(r - M) / (1.4826 * mad + 0.01)
        else:
            D = np.zeros(N)
        for i in range(N):
            if not valid[i]: continue
            full = not np.isnan(s.buf[:, i]).any()
            S = 1.0 if (full and np.ptp(s.buf[:, i]) == 0) else 0.0
            T = 1.0 if (not np.isnan(s.prev[i]) and abs(r[i] - s.prev[i]) > 0.03) else 0.0
            e[i] = min(1.0, 0.5 * min(D[i] / 3, 1) + 0.3 * S + 0.2 * T + W4 * s.dr[i])
        s.R = LAM * s.R + (1 - LAM) * np.exp(-GAM * e); s.prev = r.copy()
        ok = valid & (s.R >= RMIN); s.flags = ~ok
        if ok.sum() >= 2:
            th_hat = float((s.R[ok] * r[ok]).sum() / s.R[ok].sum())
            if abs(th_hat - s.tm) < GATE or s.dis >= 48:
                s.tm = th_hat; s.dis = 0; s.degraded = False
            else:
                s.dis += 1; s.degraded = True
        else:
            s.degraded = True
        s.v = hyst(s.v, s.tm)

class Timer:   # S1: fixed schedule, every PERIOD_D days at 06:00, depth = PERIOD_D x climatological mean ETc (no sensors)
    def __init__(s):
        s.v = 0; s.t = 0; s.left = 0
        s.depth = PERIOD_D * KC * float(ET0_SERIES.mean())
    def decide(s, r, th):
        day, step = divmod(s.t, SPD)
        if step == 24 and day % PERIOD_D == 0:
            s.left = int(round(s.depth / (RATE * DT_H)))
        s.v = 1 if s.left > 0 else 0
        if s.left > 0: s.left -= 1
    def update(s, irr, et): pass

class ET0Sched:   # S3: no sensors; water-balance model driven by the ETc estimate, hysteresis on the model
    def __init__(s): s.v = 0; s.tm = 0.28; s.t = 0
    def decide(s, r, th): s.v = hyst(s.v, s.tm)
    def update(s, irr, et): s.tm = min(FC, s.tm + (irr - et) / (1000 * ZR))

ADAPT = {"oracle_adapt", "median_adapt", "reliab_adapt", "reliab_adapt_v7"}
class Rel(Rel7):   # v8: v7 + commissioning self-calibration of the relative probe offsets
    def __init__(s):
        super().__init__()
        s.off = np.zeros(N); s.acc2 = np.zeros(N); s.n2 = 0
    def decide(s, r, th):
        just_done = False
        if s.n2 < N0:
            if not np.isnan(r).any():
                s.acc2 += r - r.mean(); s.n2 += 1
                if s.n2 == N0:
                    s.off = s.acc2 / N0; just_done = True
            rc = r
            s.R[:] = 1.0            # probes are assumed healthy while the offsets are learned: no exclusion yet
        else:
            rc = r - s.off          # a missing reading stays missing
        super().decide(rc, th)
        if just_done:               # the drift statistic restarts on offset-corrected readings
            s.base[:] = 0; s.zf[:] = 0; s.R[:] = 1.0; s.n = max(s.n, N0)

CTRL = {"oracle": Oracle, "single": Single, "median": Median, "vote": Vote, "reliab_v7": Rel7, "reliab": Rel,
        "oracle_adapt": Oracle, "median_adapt": Median, "reliab_adapt": Rel, "reliab_adapt_v7": Rel7, "s1_timer": Timer, "s3_et0": ET0Sched}
SCEN = {"none": ((), None), "stuck_low": ((0,), "low"), "stuck_high": ((0,), "high"),
        "frozen": ((0,), "frozen"), "disconnect": ((0,), "nan"), "drift": ((0,), "drift"),
        "double_stuck_low": ((0, 1), "low"),
        "drift_slow": ((0,), "drift_slow"), "drift_up": ((0,), "drift_up")}

In [ ]:
# ---------------- closed-loop simulation ----------------
def run(cname, sname, seed):
    et0 = weather_window(seed)
    etc = KC * et0
    rg = np.random.default_rng(seed + 20_000)
    g_sys = float(np.clip(1 + ET_ERR_SD * rg.normal(), 0.75, 1.25)); g_day = 1 + 0.5 * ET_ERR_SD * rg.normal(size=DAYS)
    if ET_ERR_SD == 0: g_sys, g_day = 1.0, np.ones(DAYS)
    rs = np.random.default_rng(seed)
    bias = rs.normal(0, .008, N); noise = rs.normal(0, .006, (DAYS * SPD, N))
    idx, kind = SCEN[sname]
    c = CTRL[cname](); th = 0.28
    water = drain = stress = below = below_raw = 0.0; frozen = {}; first = None; tp = fp = fn = tn = 0
    for t in range(DAYS * SPD):
        if t % SPD == 0:
            tl_ref, th_ref = adaptive_thresholds(etc[t // SPD])
            CUR["TL"], CUR["TH"] = (tl_ref, th_ref) if cname in ADAPT else (TL, TH)
        c.t = t
        r = np.clip(th + bias + noise[t], 0, 0.6)
        if t >= ONSET:
            for i in idx:
                if kind == "low": r[i] = 0.08
                elif kind == "high": r[i] = 0.45
                elif kind == "nan": r[i] = np.nan
                elif kind == "frozen": frozen.setdefault(i, r[i]); r[i] = frozen[i]
                elif kind.startswith("drift"): r[i] += {"drift": -0.008, "drift_slow": -0.003, "drift_up": 0.008}[kind] * (t - ONSET) / SPD
        c.decide(r, th)
        irr = c.v * RATE * DT_H
        h = (t % SPD) * DT_H
        frac = max(0, np.sin(np.pi * (h - 6) / 12)) * np.pi / 24 * DT_H
        et_est = etc[t // SPD] * frac
        et = et_est * g_sys * g_day[t // SPD]
        th += (irr - et) / (1000 * ZR)
        if th > FC:
            d = (th - FC) * 1000 * ZR; th = FC
        else:
            d = 0.0
        th = max(th, 0.05)
        c.update(irr, et_est)
        if t >= ONSET:
            water += irr; drain += d; stress += th < TL - 0.02; below += th < TL; below_raw += th < tl_ref
            if cname in ("reliab", "reliab_v7"):
                for i in range(N):
                    f = bool(c.flags[i]); bad = i in idx
                    tp += f and bad; fn += (not f) and bad; fp += f and (not bad); tn += (not f) and (not bad)
                if first is None and any(c.flags[i] for i in idx):
                    first = (t - ONSET) * DT_H
    n = DAYS * SPD - ONSET
    return water, drain, 100 * stress / n, first, (tp, fp, fn, tn), below * DT_H, below_raw * DT_H

# quick smoke test
print(run("reliab", "none", 0)[:3], "| window ET0 mean %.2f" % weather_window(0).mean())

## 1. Main experiment (water applied, drainage loss, time under stress, hours below the start threshold)

In [ ]:
import json, hashlib, time
def _md5(p):
    return hashlib.md5(open(p, "rb").read()).hexdigest() if os.path.exists(p) else None
CONFIG = dict(run_mode=RUN_MODE, years=YEARS, seeds=[SEED0, SEED0 + N_SEEDS - 1], days=DAYS, onset_step=ONSET,
              weather=WEATHER_SRC, weather_md5=_md5(WEATHER_CSV), n_windows=len(VALID_STARTS),
              soil=dict(FC=FC, PWP=PWP, ZR=ZR), crop=dict(p=P, Kc=KC), thresholds=dict(TL=TL, TH=TH, dth=DTH, fc_margin=FC_MARGIN),
              et_err_sd=ET_ERR_SD, s1_period_days=PERIOD_D,
              self_calibration=dict(enabled=True, n0=N0), reliability=dict(W=W, lam=LAM, gamma=GAM, Rmin=RMIN, gate=GATE), drift=dict(alpha_f=ALPHA_F, alpha_s=ALPHA_S, n0=N0, h=H_D, ratio=RATIO, w4=W4), probes=N, rate_mm_h=RATE,
              saved=time.strftime("%Y-%m-%d %H:%M:%S"))
with open(os.path.join(OUT_DIR, f"run_config_{RUN_MODE}.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)
print(json.dumps(CONFIG, indent=2))

seeds = list(range(SEED0, SEED0 + N_SEEDS))
print("WEATHER:", WEATHER_SRC, "| DAYS =", DAYS, "| ET_ERR_SD =", ET_ERR_SD, "| seeds", seeds[0], "-", seeds[-1])
res, rows, per_seed = {}, [], []
for sc in SCEN:
    for c in CTRL:
        out = [run(c, sc, k) for k in seeds]
        w, d, st = (np.array([o[i] for o in out]) for i in range(3))
        bh, br = np.array([o[5] for o in out]), np.array([o[6] for o in out])
        res[(sc, c)] = (w, d, st, bh, br)
        rows.append([sc, c, w.mean(), w.std(), d.mean(), d.std(), np.percentile(d, 95), float((d > 5).mean()), st.mean(), bh.mean(), bh.std(), br.mean(), br.std()])
        for k, o in zip(seeds, out):
            per_seed.append([sc, c, k, o[0], o[1], o[2], o[5], o[6]])
cols = ["scenario", "ctrl", "water_mean", "water_sd", "drain_mean", "drain_sd", "drain_p95", "share_drain_gt5", "stress_mean", "below_TL_h_mean", "below_TL_h_sd", "below_RAW_h_mean", "below_RAW_h_sd"]
with open(os.path.join(OUT_DIR, f"per_seed_{RUN_MODE}.csv"), "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(["scenario", "ctrl", "seed", "water", "drain", "stress", "below_TL_h", "below_RAW_h"]); wr.writerows(per_seed)
with open(os.path.join(OUT_DIR, f"sim_results_{RUN_MODE}_seed{SEED0}.csv"), "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(cols); wr.writerows(rows)
if pd is not None:
    table = pd.DataFrame(rows, columns=cols).round(2)
    display(table) if "display" in globals() else print(table.to_string(index=False))
else:
    for r in rows: print(r)

## 2. Paired statistics: reliab minus median/vote (negative = reliab uses less)

In [ ]:
def cliffs_delta(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return float(((a[:, None] > b[None, :]).sum() - (a[:, None] < b[None, :]).sum()) / (len(a) * len(b)))

def sign_test_p(d):
    d = d[d != 0]; n = len(d)
    if n == 0: return 1.0
    k = int((d > 0).sum()); m = min(k, n - k)
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(m + 1)) / 2 ** n)

def paired(a, b, rng, B=10000):
    d = np.asarray(a) - np.asarray(b)
    bs = np.array([rng.choice(d, len(d), replace=True).mean() for _ in range(B)])
    if np.all(d == 0): pv = 1.0
    elif HAVE_SCIPY: pv = float(wilcoxon(d).pvalue)
    else: pv = sign_test_p(d)
    return d.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5), pv

METRIC = {"water": 0, "drain": 1, "below_TL_h": 3, "below_RAW_h": 4}
# (A, B, metrics): negative mean_diff = A lower than B
COMPARISONS = [("reliab", b, ("water", "below_TL_h")) for b in ("median", "vote", "reliab_v7")] +               [("reliab_adapt", "reliab_adapt_v7", ("water", "drain", "below_RAW_h")),  # effect of self-calibration under adaptive thresholds
               ("reliab_adapt", "reliab", ("water", "below_RAW_h")),        # effect of adaptive thresholds (fusion held fixed)
               ("reliab_adapt", "median_adapt", ("water", "below_RAW_h")),  # effect of fusion under adaptive thresholds
               ("reliab_adapt", "s3_et0", ("water", "below_RAW_h")),
               ("reliab_adapt", "s1_timer", ("water", "below_RAW_h")),
               ("reliab", "median", ("below_RAW_h",))]                      # fixed-threshold systems scored against the FAO-56 threshold
rng = np.random.default_rng(7)
stat_rows = []
for sc in SCEN:
    for A, B_, mets in COMPARISONS:
        for name in mets:
            k = METRIC[name]
            m, lo, hi, pv = paired(res[(sc, A)][k], res[(sc, B_)][k], rng)
            stat_rows.append([sc, A, B_, name, m, lo, hi, pv, cliffs_delta(res[(sc, A)][k], res[(sc, B_)][k])])
scols = ["scenario", "A", "B", "metric", "mean_diff_A_minus_B", "ci_lo", "ci_hi", "p_uncorrected", "cliffs_delta"]
m_tests = len(stat_rows)
print(f"p-values are uncorrected ({'Wilcoxon' if HAVE_SCIPY else 'sign test'}); Bonferroni family size = {m_tests}")
if pd is not None:
    st_df = pd.DataFrame(stat_rows, columns=scols)
    st_df["p_bonferroni"] = (st_df["p_uncorrected"] * m_tests).clip(upper=1)
    st_df.round(4).to_csv(os.path.join(OUT_DIR, f"paired_stats_{RUN_MODE}.csv"), index=False)
    sig = st_df[st_df.p_bonferroni < 0.05]
    print("Comparisons significant after Bonferroni:")
    display(sig.round(3)) if "display" in globals() else print(sig.round(3).to_string(index=False))
    print("Full table saved to", os.path.join(OUT_DIR, f"paired_stats_{RUN_MODE}.csv"))
else:
    for r in stat_rows: print([round(x, 3) if isinstance(x, float) else x for x in r])

## 3. Fault detection (reliab): per sensor-step precision / recall / F1, delay, false-alarm rate

In [ ]:
det_rows = []
for sc in SCEN:
    for cn in ("reliab_v7", "reliab"):
        out = [run(cn, sc, k) for k in seeds]
        tp, fp, fn, tn = (sum(o[4][i] for o in out) for i in range(4))
        pr = tp / (tp + fp) if tp + fp else float("nan"); rc = tp / (tp + fn) if tp + fn else float("nan")
        f1 = 2 * pr * rc / (pr + rc) if (pr == pr and rc == rc and pr + rc > 0) else float("nan")
        dl = [o[3] for o in out if o[3] is not None]
        det_rows.append([sc, cn, fp / (fp + tn), pr, rc, f1, float(np.mean(dl)) if dl else float("nan"), f"{len(dl)}/{len(out)}"])
dcols = ["scenario", "controller", "FPR", "precision", "recall", "F1", "delay_h", "detected"]
if pd is not None:
    det_df = pd.DataFrame(det_rows, columns=dcols).round(3)
    det_df.to_csv(os.path.join(OUT_DIR, f"detection_{RUN_MODE}.csv"), index=False)
    display(det_df) if "display" in globals() else print(det_df.to_string(index=False))
else:
    for r in det_rows: print(r)

## 4. Figure: mean water applied per scenario (log scale)

In [ ]:
if plt is not None:
    names = list(SCEN); ctrls = ["oracle", "median", "vote", "reliab"]
    x = np.arange(len(names)); wdt = 0.2
    fig, ax = plt.subplots(figsize=(10, 4))
    for j, c in enumerate(ctrls):
        ax.bar(x + (j - 1.5) * wdt, [res[(sc, c)][0].mean() for sc in names], wdt, label=c)
    ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(names, rotation=20)
    ax.set_ylabel("water applied after fault onset (mm)"); ax.legend(); ax.set_title("Fault scenarios, 3 probes per zone")
    fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, f"water_by_scenario_{RUN_MODE}.png"), dpi=150); plt.show()
else:
    print("matplotlib not available; figure skipped")

## 5. Cross-zone single-valve rule (old design): the reviewer's counterexample

In [ ]:
for lab, m in [("zone A truly dry, no fault", [0.22, 0.30, 0.30]),
               ("zone A dry, zone C probe stuck-low", [0.22, 0.30, 0.08])]:
    nL = sum(x <= TL for x in m)
    print(f"{lab}: readings={m} nL={nL} -> valve {'OPEN' if nL >= len(m) / 2 else 'stays closed'}")

## 6. Export the run outputs (added after the frozen code; it does not change any result)
Creates `sim_outputs_<mode>.zip` with `run_config`, `sim_results`, `per_seed`, `paired_stats`, `detection` and the figure. Run it once the cells above have finished, and download the zip.

In [ ]:
import os, shutil
out_dir = globals().get("OUT_DIR", "sim_outputs")
mode = globals().get("RUN_MODE", "dev")
if os.path.isdir(out_dir):
    name = f"sim_outputs_{mode}"
    shutil.make_archive(name, "zip", out_dir)
    print("Created", name + ".zip", "with:", sorted(os.listdir(out_dir)))
    try:
        from google.colab import files      # Colab: start the download
        files.download(name + ".zip")
    except Exception:
        print("Download the zip from the file panel (Kaggle: Output / working directory; Jupyter: the notebook folder).")
else:
    print("Output folder not found:", out_dir, "- run the cells above first.")